# Chapter 12.1: Disaggregated Serving -- Mooncake, DistServe, Splitwise

## Overview

Disaggregated serving is one of the most impactful architectural innovations in LLM inference. Instead of running both **prefill** (prompt processing) and **decode** (token generation) on the same GPU, disaggregated architectures split these phases across separate hardware pools.

### Key Papers
- **DistServe** (OSDI 2024): Disaggregating prefill and decoding for goodput-optimized LLM serving
- **Splitwise** (ISCA 2024): Efficient generative LLM serving with phase splitting
- **Mooncake** (2024): A KVCache-centric disaggregated architecture for LLM serving

### Why Disaggregate?
| Aspect | Prefill Phase | Decode Phase |
|--------|--------------|-------------|
| Compute | Compute-bound | Memory-bound |
| Parallelism | High (full prompt) | Low (one token) |
| Latency goal | TTFT (Time to First Token) | TPOT (Time per Output Token) |
| GPU utilization | High FLOPs | Low FLOPs, high bandwidth |

**Learning Objectives:**
1. Understand prefill/decode phase characteristics
2. Analyze KV-cache transfer mechanisms
3. Simulate disaggregated vs co-located serving
4. Calculate optimal prefill/decode node ratios
5. Implement scheduling and transfer simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part12/chapter_12.1_disaggregated.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part12/chapter_12.1_disaggregated.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import heapq
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import deque
import json

np.random.seed(42)
plt.style.use('default')
print("Environment ready for disaggregated serving analysis.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_disaggregated_vs_colocated():
    """Side by side: Co-located (same GPU) vs Disaggregated (Prefill GPU -> KV transfer -> Decode GPU)."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    # --- Co-located (Left) ---
    ax1.set_xlim(0, 10); ax1.set_ylim(0, 10); ax1.axis('off')
    ax1.set_title('Co-located Serving\n(Same GPU does Prefill + Decode)',
                  fontsize=13, fontweight='bold', color='#4A90D9', pad=10)

    # Single GPU box
    gpu_box = FancyBboxPatch((1, 1), 8, 7.5, boxstyle='round,pad=0.2',
                             facecolor='#E3F2FD', edgecolor='#4A90D9', linewidth=2)
    ax1.add_patch(gpu_box)
    ax1.text(5, 9, 'GPU 0', ha='center', fontsize=12, fontweight='bold', color='#4A90D9')

    # Prefill phase
    pf = FancyBboxPatch((1.5, 5.5), 7, 2.5, boxstyle='round,pad=0.1',
                        facecolor='#E74C3C', edgecolor='white', linewidth=2, alpha=0.85)
    ax1.add_patch(pf)
    ax1.text(5, 6.75, 'Prefill Phase\n(Compute-Bound)\nProcess all prompt tokens', ha='center', va='center',
             fontsize=10, fontweight='bold', color='white')

    # Decode phase
    dc = FancyBboxPatch((1.5, 1.5), 7, 3.5, boxstyle='round,pad=0.1',
                        facecolor='#4A90D9', edgecolor='white', linewidth=2, alpha=0.85)
    ax1.add_patch(dc)
    ax1.text(5, 3.25, 'Decode Phase\n(Memory-Bound)\nGenerate tokens one by one\n\nGPU underutilized!',
             ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    # Arrow between phases
    ax1.annotate('', xy=(5, 5.5), xytext=(5, 5.0),
                arrowprops=dict(arrowstyle='<-', color='#333', lw=2))
    ax1.text(5.5, 5.15, 'sequential', fontsize=8, color='#666')

    # --- Disaggregated (Right) ---
    ax2.set_xlim(0, 14); ax2.set_ylim(0, 10); ax2.axis('off')
    ax2.set_title('Disaggregated Serving\n(Separate Prefill & Decode GPUs)',
                  fontsize=13, fontweight='bold', color='#27AE60', pad=10)

    # Prefill GPU
    pf_gpu = FancyBboxPatch((0.5, 2), 4.5, 6.5, boxstyle='round,pad=0.15',
                            facecolor='#FFEBEE', edgecolor='#E74C3C', linewidth=2)
    ax2.add_patch(pf_gpu)
    ax2.text(2.75, 9, 'Prefill GPU', ha='center', fontsize=11, fontweight='bold', color='#E74C3C')

    pf_inner = FancyBboxPatch((1, 3), 3.5, 5, boxstyle='round,pad=0.1',
                              facecolor='#E74C3C', edgecolor='white', linewidth=2, alpha=0.85)
    ax2.add_patch(pf_inner)
    ax2.text(2.75, 5.5, 'Prefill Phase\n(Compute-Bound)\n\nHigh GPU\nutilization!',
             ha='center', va='center', fontsize=9, fontweight='bold', color='white')

    # KV Transfer arrow
    ax2.annotate('', xy=(9, 5.5), xytext=(5.5, 5.5),
                arrowprops=dict(arrowstyle='->', color='#F39C12', lw=3))
    ax2.text(7.25, 6.2, 'KV-Cache\nTransfer', ha='center', fontsize=9,
             fontweight='bold', color='#F39C12',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFF3E0', edgecolor='#F39C12'))

    # Decode GPU
    dc_gpu = FancyBboxPatch((9, 2), 4.5, 6.5, boxstyle='round,pad=0.15',
                            facecolor='#E8F5E9', edgecolor='#27AE60', linewidth=2)
    ax2.add_patch(dc_gpu)
    ax2.text(11.25, 9, 'Decode GPU', ha='center', fontsize=11, fontweight='bold', color='#27AE60')

    dc_inner = FancyBboxPatch((9.5, 3), 3.5, 5, boxstyle='round,pad=0.1',
                              facecolor='#4A90D9', edgecolor='white', linewidth=2, alpha=0.85)
    ax2.add_patch(dc_inner)
    ax2.text(11.25, 5.5, 'Decode Phase\n(Memory-Bound)\n\nOptimized for\nbandwidth!',
             ha='center', va='center', fontsize=9, fontweight='bold', color='white')

    plt.tight_layout()
    plt.savefig('disaggregated_vs_colocated.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_disaggregated_vs_colocated()

## 1. Prefill vs Decode Phase Analysis

The fundamental insight behind disaggregation: prefill is **compute-bound** while decode is **memory-bandwidth-bound**. They have fundamentally different hardware requirements.

In [ ]:
# Model and hardware parameters
@dataclass
class ModelConfig:
    name: str
    num_layers: int
    hidden_dim: int
    num_heads: int
    head_dim: int
    vocab_size: int
    param_bytes: int = 2  # FP16

    @property
    def total_params_gb(self):
        # Approximate: 12 * L * d^2 parameters (attention + FFN)
        params = 12 * self.num_layers * self.hidden_dim ** 2
        return params * self.param_bytes / (1024**3)

    @property
    def kv_cache_per_token_bytes(self):
        # 2 (K and V) * num_layers * num_heads * head_dim * param_bytes
        return 2 * self.num_layers * self.num_heads * self.head_dim * self.param_bytes

@dataclass
class GPUConfig:
    name: str
    memory_gb: float
    compute_tflops: float  # FP16 TFLOPS
    memory_bandwidth_gbps: float  # GB/s

    @property
    def arithmetic_intensity_threshold(self):
        """ops/byte threshold: above this = compute-bound, below = memory-bound"""
        return self.compute_tflops * 1e12 / (self.memory_bandwidth_gbps * 1e9)

# Define common configurations
llama_70b = ModelConfig("LLaMA-70B", 80, 8192, 64, 128, 32000)
llama_13b = ModelConfig("LLaMA-13B", 40, 5120, 40, 128, 32000)
a100 = GPUConfig("A100-80GB", 80, 312, 2039)
h100 = GPUConfig("H100-80GB", 80, 989, 3350)

print(f"Model: {llama_70b.name}")
print(f"  Approx parameters: {llama_70b.total_params_gb:.1f} GB (FP16)")
print(f"  KV-cache per token: {llama_70b.kv_cache_per_token_bytes / 1024:.1f} KB")
print(f"\nGPU: {h100.name}")
print(f"  Arithmetic intensity threshold: {h100.arithmetic_intensity_threshold:.0f} ops/byte")
print(f"  Above this => compute-bound, below => memory-bound")

In [ ]:
def compute_phase_characteristics(model: ModelConfig, gpu: GPUConfig,
                                   prompt_len: int, batch_size: int = 1):
    """Compute the arithmetic intensity and time for prefill vs decode."""
    # Prefill: process all prompt tokens at once
    # FLOPs ~ 2 * num_params * prompt_len * batch_size (forward pass)
    total_params = 12 * model.num_layers * model.hidden_dim ** 2
    prefill_flops = 2 * total_params * prompt_len * batch_size
    # Bytes loaded: model weights + KV-cache writes
    prefill_bytes = total_params * model.param_bytes  # load weights once
    prefill_ai = prefill_flops / prefill_bytes  # arithmetic intensity
    prefill_time_compute = prefill_flops / (gpu.compute_tflops * 1e12)
    prefill_time_memory = prefill_bytes / (gpu.memory_bandwidth_gbps * 1e9)
    prefill_time = max(prefill_time_compute, prefill_time_memory)

    # Decode: one token at a time per sequence
    decode_flops = 2 * total_params * 1 * batch_size
    decode_bytes = total_params * model.param_bytes  # load weights
    decode_ai = decode_flops / decode_bytes
    decode_time_compute = decode_flops / (gpu.compute_tflops * 1e12)
    decode_time_memory = decode_bytes / (gpu.memory_bandwidth_gbps * 1e9)
    decode_time = max(decode_time_compute, decode_time_memory)

    return {
        'prefill_ai': prefill_ai, 'decode_ai': decode_ai,
        'prefill_time_s': prefill_time, 'decode_time_per_token_s': decode_time,
        'prefill_compute_bound': prefill_time_compute > prefill_time_memory,
        'decode_compute_bound': decode_time_compute > decode_time_memory,
        'prefill_gpu_util': min(1.0, prefill_ai / gpu.arithmetic_intensity_threshold),
        'decode_gpu_util': min(1.0, decode_ai / gpu.arithmetic_intensity_threshold),
    }

# Analyze for various prompt lengths
prompt_lengths = [128, 256, 512, 1024, 2048, 4096]
batch_sizes = [1, 4, 16, 64]

print(f"{'Prompt Len':>10} {'Batch':>6} {'Prefill AI':>10} {'Decode AI':>10} "
      f"{'Prefill Bound':>14} {'Decode Bound':>13} {'Prefill Util':>12} {'Decode Util':>12}")
print("-" * 100)
for pl in [512, 2048]:
    for bs in batch_sizes:
        r = compute_phase_characteristics(llama_70b, h100, pl, bs)
        print(f"{pl:>10} {bs:>6} {r['prefill_ai']:>10.1f} {r['decode_ai']:>10.1f} "
              f"{'Compute':>14} {'Compute' if r['decode_compute_bound'] else 'Memory':>13} "
              f"{r['prefill_gpu_util']:>11.1%} {r['decode_gpu_util']:>11.1%}")

In [ ]:
# Visualize the compute vs memory bound regions (Roofline model)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, gpu in zip(axes, [a100, h100]):
    ai_range = np.logspace(-1, 4, 500)
    # Roofline: perf = min(compute_peak, bandwidth * AI)
    perf = np.minimum(gpu.compute_tflops, gpu.memory_bandwidth_gbps * 1e-3 * ai_range)
    ax.loglog(ai_range, perf, 'b-', linewidth=2, label='Roofline')

    # Mark prefill and decode points
    for bs_val, marker in [(1, 'o'), (16, 's'), (64, '^')]:
        for pl, color in [(512, 'green'), (2048, 'red')]:
            r = compute_phase_characteristics(llama_70b, gpu_config := gpu, pl, bs_val)
            # Prefill
            pf_perf = min(gpu.compute_tflops, gpu.memory_bandwidth_gbps * 1e-3 * r['prefill_ai'])
            ax.plot(r['prefill_ai'], pf_perf, marker, color=color, markersize=8,
                   label=f'Prefill pl={pl} bs={bs_val}' if pl == 512 else '')
            # Decode
            dc_perf = min(gpu.compute_tflops, gpu.memory_bandwidth_gbps * 1e-3 * r['decode_ai'])
            ax.plot(r['decode_ai'], dc_perf, marker, color='purple', markersize=8,
                   markerfacecolor='none', linewidth=2,
                   label=f'Decode bs={bs_val}' if pl == 512 else '')

    ax.axvline(gpu.arithmetic_intensity_threshold, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Arithmetic Intensity (FLOPs/Byte)')
    ax.set_ylabel('Performance (TFLOPS)')
    ax.set_title(f'{gpu.name} Roofline')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roofline_prefill_decode.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: Prefill operates in the compute-bound region, decode in the memory-bound region.")

## 2. Paper Deep Dive: DistServe, Splitwise, Mooncake

### DistServe (OSDI 2024)
- **Key Idea**: Assign prefill and decode to separate GPU instances
- **Goodput optimization**: Maximizes the number of requests meeting SLO targets
- **Placement algorithm**: Determines optimal GPU allocation for prefill vs decode

### Splitwise (ISCA 2024)
- **Key Idea**: Split prompt and token phases across machines optimized for each
- **Mixed clusters**: Use compute-optimized GPUs for prefill, memory-optimized for decode
- **KV-cache migration**: Efficient transfer of intermediate state

### Mooncake
- **Key Idea**: KVCache-centric architecture with disaggregated KV-cache pool
- **KVCache as shared storage**: Prefill nodes write, decode nodes read
- **RDMA transfers**: Low-latency KV-cache migration

In [ ]:
# Paper comparison table
papers = {
    'DistServe': {
        'venue': 'OSDI 2024',
        'key_innovation': 'Goodput-optimized disaggregation with placement algorithm',
        'transfer_mechanism': 'NCCL / direct GPU-GPU',
        'scheduling': 'SLO-aware with separate prefill/decode queues',
        'reported_improvement': '1.5-4.5x goodput improvement',
        'kv_cache_handling': 'Direct transfer after prefill',
    },
    'Splitwise': {
        'venue': 'ISCA 2024',
        'key_innovation': 'Mixed-cluster with heterogeneous hardware',
        'transfer_mechanism': 'InfiniBand / RDMA',
        'scheduling': 'Phase-aware scheduling',
        'reported_improvement': '1.4x throughput at same SLO',
        'kv_cache_handling': 'Pipelined transfer overlapped with computation',
    },
    'Mooncake': {
        'venue': 'Arxiv 2024',
        'key_innovation': 'KVCache-centric with shared KV pool',
        'transfer_mechanism': 'RDMA with KVCache store',
        'scheduling': 'Prefix-aware with KV reuse',
        'reported_improvement': 'Up to 525% throughput improvement',
        'kv_cache_handling': 'Shared KVCache pool with prefix matching',
    }
}

for name, info in papers.items():
    print(f"\n{'='*60}")
    print(f" {name} ({info['venue']})")
    print(f"{'='*60}")
    for k, v in info.items():
        if k != 'venue':
            print(f"  {k.replace('_', ' ').title():30s}: {v}")

## 3. Demo: KV-Cache Transfer Simulator

A critical component of disaggregated serving is the KV-cache transfer from prefill nodes to decode nodes. Let's simulate different transfer strategies.

In [ ]:
class KVCacheTransferSimulator:
    """Simulates KV-cache transfer between prefill and decode nodes."""

    def __init__(self, model: ModelConfig):
        self.model = model

    def kv_cache_size_bytes(self, seq_len: int) -> int:
        """Total KV-cache size for a sequence."""
        return self.model.kv_cache_per_token_bytes * seq_len

    def transfer_time(self, seq_len: int, bandwidth_gbps: float) -> float:
        """Time to transfer KV-cache via network."""
        size_bytes = self.kv_cache_size_bytes(seq_len)
        return size_bytes / (bandwidth_gbps * 1e9)

    def recompute_time(self, seq_len: int, gpu: GPUConfig) -> float:
        """Time to recompute KV-cache from scratch on decode node."""
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        flops = 2 * total_params * seq_len
        return flops / (gpu.compute_tflops * 1e12)

    def pipelined_transfer_time(self, seq_len: int, bandwidth_gbps: float,
                                 num_chunks: int) -> float:
        """Pipelined transfer: overlap transfer of chunk i with decode using chunk i-1."""
        chunk_size = seq_len // num_chunks
        chunk_transfer_time = self.transfer_time(chunk_size, bandwidth_gbps)
        # First chunk must be fully transferred; rest overlap with computation
        return chunk_transfer_time  # effective latency (rest is pipelined)

# Simulate transfer times
sim = KVCacheTransferSimulator(llama_70b)

seq_lengths = [256, 512, 1024, 2048, 4096, 8192]
bandwidths = {
    'PCIe 4.0 x16': 31.5,      # GB/s
    'NVLink': 600,               # GB/s (H100)
    'InfiniBand HDR': 25,        # GB/s
    'RDMA RoCE 100G': 12.5,      # GB/s
    'Ethernet 100G': 12.5,       # GB/s
}

print(f"KV-Cache sizes (LLaMA-70B):")
print(f"{'Seq Len':>10} {'KV-Cache Size':>15}")
for sl in seq_lengths:
    size = sim.kv_cache_size_bytes(sl)
    print(f"{sl:>10} {size / 1024**2:>12.1f} MB")

print(f"\nTransfer times (ms):")
print(f"{'Seq Len':>10}", end='')
for name in bandwidths:
    print(f"{name:>20}", end='')
print(f"{'Recompute(H100)':>18}")
print("-" * 110)
for sl in seq_lengths:
    print(f"{sl:>10}", end='')
    for name, bw in bandwidths.items():
        t = sim.transfer_time(sl, bw) * 1000
        print(f"{t:>18.2f}ms", end='')
    recomp = sim.recompute_time(sl, h100) * 1000
    print(f"{recomp:>16.2f}ms")

In [ ]:
# Visualize: Transfer time vs Recomputation time
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

seq_range = np.arange(128, 16385, 128)
recomp_times = [sim.recompute_time(s, h100) * 1000 for s in seq_range]

ax1.plot(seq_range, recomp_times, 'k--', linewidth=2, label='Recompute (H100)')
for name, bw in bandwidths.items():
    transfer_times = [sim.transfer_time(s, bw) * 1000 for s in seq_range]
    ax1.plot(seq_range, transfer_times, linewidth=1.5, label=name)

ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Transfer vs Recomputation Time')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Crossover analysis: when does transfer become cheaper than recomputation?
for name, bw in bandwidths.items():
    crossover_points = []
    for s in seq_range:
        transfer_t = sim.transfer_time(s, bw)
        recomp_t = sim.recompute_time(s, h100)
        crossover_points.append(transfer_t / recomp_t)
    ax2.plot(seq_range, crossover_points, linewidth=1.5, label=name)

ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='Breakeven')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Transfer / Recompute Ratio')
ax2.set_title('Transfer vs Recompute Cost Ratio')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 5)

plt.tight_layout()
plt.savefig('transfer_vs_recompute.png', dpi=150, bbox_inches='tight')
plt.show()
print("Below the red line = transfer is cheaper than recomputation.")

## 4. Demo: Disaggregated vs Co-Located Serving Simulation

Let's build a full simulation comparing co-located (traditional) serving with disaggregated serving.

In [ ]:
@dataclass
class Request:
    id: int
    arrival_time: float
    prompt_len: int
    output_len: int
    # Timestamps filled during simulation
    prefill_start: float = 0.0
    prefill_end: float = 0.0
    decode_start: float = 0.0
    decode_end: float = 0.0

    @property
    def ttft(self):
        return self.prefill_end - self.arrival_time

    @property
    def tpot(self):
        if self.output_len <= 1:
            return 0
        return (self.decode_end - self.decode_start) / self.output_len

    @property
    def total_latency(self):
        return self.decode_end - self.arrival_time


def generate_workload(num_requests: int, rate: float,
                      prompt_len_range=(256, 2048),
                      output_len_range=(32, 512)) -> List[Request]:
    """Generate a Poisson workload."""
    requests = []
    current_time = 0.0
    for i in range(num_requests):
        inter_arrival = np.random.exponential(1.0 / rate)
        current_time += inter_arrival
        prompt_len = np.random.randint(*prompt_len_range)
        output_len = np.random.randint(*output_len_range)
        requests.append(Request(id=i, arrival_time=current_time,
                                prompt_len=prompt_len, output_len=output_len))
    return requests

workload = generate_workload(200, rate=10)  # 10 requests/second
print(f"Generated {len(workload)} requests")
print(f"Avg prompt length: {np.mean([r.prompt_len for r in workload]):.0f}")
print(f"Avg output length: {np.mean([r.output_len for r in workload]):.0f}")
print(f"Time span: {workload[0].arrival_time:.2f}s - {workload[-1].arrival_time:.2f}s")

In [ ]:
class CoLocatedServer:
    """Traditional co-located serving: prefill and decode share the same GPU."""

    def __init__(self, num_gpus: int, model: ModelConfig, gpu: GPUConfig,
                 max_batch_size: int = 32):
        self.num_gpus = num_gpus
        self.model = model
        self.gpu = gpu
        self.max_batch_size = max_batch_size
        # Each GPU tracks its next available time
        self.gpu_available = [0.0] * num_gpus

    def _prefill_time(self, prompt_len: int) -> float:
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        flops = 2 * total_params * prompt_len
        return flops / (self.gpu.compute_tflops * 1e12)

    def _decode_time_per_token(self, batch_size: int = 1) -> float:
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        bytes_to_load = total_params * self.model.param_bytes
        return bytes_to_load / (self.gpu.memory_bandwidth_gbps * 1e9)

    def serve(self, requests: List[Request]) -> List[Request]:
        """Simple FCFS scheduling on co-located GPUs."""
        import copy
        results = [copy.deepcopy(r) for r in requests]

        for req in results:
            # Find GPU with earliest available time
            gpu_idx = np.argmin(self.gpu_available)
            start_time = max(req.arrival_time, self.gpu_available[gpu_idx])

            # Prefill
            req.prefill_start = start_time
            prefill_t = self._prefill_time(req.prompt_len)
            req.prefill_end = start_time + prefill_t

            # Decode (blocking during decode = GPU busy)
            req.decode_start = req.prefill_end
            decode_t = self._decode_time_per_token() * req.output_len
            req.decode_end = req.decode_start + decode_t

            self.gpu_available[gpu_idx] = req.decode_end

        return results


class DisaggregatedServer:
    """Disaggregated serving: separate prefill and decode GPU pools."""

    def __init__(self, num_prefill_gpus: int, num_decode_gpus: int,
                 model: ModelConfig, gpu: GPUConfig, transfer_bw_gbps: float = 25.0):
        self.num_prefill = num_prefill_gpus
        self.num_decode = num_decode_gpus
        self.model = model
        self.gpu = gpu
        self.transfer_bw = transfer_bw_gbps
        self.prefill_available = [0.0] * num_prefill_gpus
        self.decode_available = [0.0] * num_decode_gpus
        self.transfer_sim = KVCacheTransferSimulator(model)

    def _prefill_time(self, prompt_len: int) -> float:
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        flops = 2 * total_params * prompt_len
        return flops / (self.gpu.compute_tflops * 1e12)

    def _decode_time_per_token(self) -> float:
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        bytes_to_load = total_params * self.model.param_bytes
        return bytes_to_load / (self.gpu.memory_bandwidth_gbps * 1e9)

    def serve(self, requests: List[Request]) -> List[Request]:
        import copy
        results = [copy.deepcopy(r) for r in requests]

        for req in results:
            # Assign to prefill GPU
            pf_idx = np.argmin(self.prefill_available)
            pf_start = max(req.arrival_time, self.prefill_available[pf_idx])
            req.prefill_start = pf_start
            prefill_t = self._prefill_time(req.prompt_len)
            req.prefill_end = pf_start + prefill_t
            self.prefill_available[pf_idx] = req.prefill_end  # prefill GPU freed

            # Transfer KV-cache
            transfer_t = self.transfer_sim.transfer_time(req.prompt_len, self.transfer_bw)

            # Assign to decode GPU
            dc_idx = np.argmin(self.decode_available)
            dc_start = max(req.prefill_end + transfer_t, self.decode_available[dc_idx])
            req.decode_start = dc_start
            decode_t = self._decode_time_per_token() * req.output_len
            req.decode_end = dc_start + decode_t
            self.decode_available[dc_idx] = req.decode_end

        return results


# Run both simulations
colocated = CoLocatedServer(4, llama_70b, h100)
disagg = DisaggregatedServer(2, 2, llama_70b, h100, transfer_bw_gbps=25.0)

colocated_results = colocated.serve(workload)
disagg_results = disagg.serve(workload)

# Metrics
def compute_metrics(results):
    ttfts = [r.ttft for r in results]
    tpots = [r.tpot for r in results]
    latencies = [r.total_latency for r in results]
    throughput = len(results) / (results[-1].decode_end - results[0].arrival_time)
    return {
        'avg_ttft': np.mean(ttfts),
        'p99_ttft': np.percentile(ttfts, 99),
        'avg_tpot': np.mean(tpots),
        'avg_latency': np.mean(latencies),
        'throughput': throughput,
    }

cm = compute_metrics(colocated_results)
dm = compute_metrics(disagg_results)

print(f"{'Metric':25s} {'Co-Located':>15s} {'Disaggregated':>15s} {'Improvement':>12s}")
print("-" * 70)
for key in cm:
    c_val = cm[key]
    d_val = dm[key]
    if 'throughput' in key:
        improvement = f"{d_val/c_val:.2f}x"
    else:
        improvement = f"{(c_val - d_val)/c_val:.1%}"
    unit = 'req/s' if 'throughput' in key else 's'
    print(f"{key:25s} {c_val:>13.4f}{unit:>2s} {d_val:>13.4f}{unit:>2s} {improvement:>12s}")

In [ ]:
# Visualize latency distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ttft_co = [r.ttft * 1000 for r in colocated_results]
ttft_dis = [r.ttft * 1000 for r in disagg_results]
axes[0].hist(ttft_co, bins=40, alpha=0.6, label='Co-Located', density=True)
axes[0].hist(ttft_dis, bins=40, alpha=0.6, label='Disaggregated', density=True)
axes[0].set_xlabel('TTFT (ms)')
axes[0].set_ylabel('Density')
axes[0].set_title('Time to First Token')
axes[0].legend()

tpot_co = [r.tpot * 1000 for r in colocated_results if r.tpot > 0]
tpot_dis = [r.tpot * 1000 for r in disagg_results if r.tpot > 0]
axes[1].hist(tpot_co, bins=40, alpha=0.6, label='Co-Located', density=True)
axes[1].hist(tpot_dis, bins=40, alpha=0.6, label='Disaggregated', density=True)
axes[1].set_xlabel('TPOT (ms)')
axes[1].set_title('Time per Output Token')
axes[1].legend()

lat_co = [r.total_latency for r in colocated_results]
lat_dis = [r.total_latency for r in disagg_results]
axes[2].hist(lat_co, bins=40, alpha=0.6, label='Co-Located', density=True)
axes[2].hist(lat_dis, bins=40, alpha=0.6, label='Disaggregated', density=True)
axes[2].set_xlabel('Total Latency (s)')
axes[2].set_title('End-to-End Latency')
axes[2].legend()

plt.tight_layout()
plt.savefig('disagg_vs_colocated.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Demo: Optimal Prefill/Decode Node Ratio Calculator

Given a fixed GPU budget, what is the optimal split between prefill and decode nodes?

In [ ]:
def find_optimal_ratio(total_gpus: int, workload: List[Request],
                        model: ModelConfig, gpu: GPUConfig,
                        transfer_bw: float = 25.0) -> dict:
    """Try all possible prefill/decode splits and find the optimal one."""
    results = []

    for num_prefill in range(1, total_gpus):
        num_decode = total_gpus - num_prefill
        server = DisaggregatedServer(num_prefill, num_decode, model, gpu, transfer_bw)
        served = server.serve(workload)
        metrics = compute_metrics(served)
        metrics['num_prefill'] = num_prefill
        metrics['num_decode'] = num_decode
        metrics['ratio'] = f"{num_prefill}:{num_decode}"
        results.append(metrics)

    return results

# Test with 8 total GPUs
total_gpus = 8
ratio_results = find_optimal_ratio(total_gpus, workload, llama_70b, h100)

print(f"{'Ratio (P:D)':>12} {'Throughput':>12} {'Avg TTFT':>12} {'P99 TTFT':>12} {'Avg Latency':>12}")
print("-" * 65)
for r in ratio_results:
    print(f"{r['ratio']:>12} {r['throughput']:>10.2f}/s {r['avg_ttft']*1000:>10.1f}ms "
          f"{r['p99_ttft']*1000:>10.1f}ms {r['avg_latency']:>10.3f}s")

# Find optimal by throughput
best = max(ratio_results, key=lambda x: x['throughput'])
print(f"\nOptimal ratio for throughput: {best['ratio']} "
      f"({best['throughput']:.2f} req/s)")

In [ ]:
# Visualize the optimization landscape
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ratios = [r['num_prefill'] for r in ratio_results]

axes[0].bar(ratios, [r['throughput'] for r in ratio_results], color='steelblue')
axes[0].set_xlabel('Number of Prefill GPUs')
axes[0].set_ylabel('Throughput (req/s)')
axes[0].set_title(f'Throughput vs Prefill GPUs (Total={total_gpus})')

axes[1].bar(ratios, [r['avg_ttft']*1000 for r in ratio_results], color='coral')
axes[1].set_xlabel('Number of Prefill GPUs')
axes[1].set_ylabel('Avg TTFT (ms)')
axes[1].set_title('TTFT vs Prefill GPUs')

axes[2].bar(ratios, [r['avg_latency'] for r in ratio_results], color='green')
axes[2].set_xlabel('Number of Prefill GPUs')
axes[2].set_ylabel('Avg Latency (s)')
axes[2].set_title('Latency vs Prefill GPUs')

plt.tight_layout()
plt.savefig('optimal_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Demo: Mooncake-Style Prefix-Aware KV-Cache Reuse

Mooncake introduces a shared KVCache pool with prefix matching. When multiple requests share the same system prompt prefix, the KV-cache can be reused rather than recomputed.

In [ ]:
class KVCachePool:
    """Mooncake-style shared KV-Cache pool with prefix matching."""

    def __init__(self, capacity_tokens: int, model: ModelConfig):
        self.capacity = capacity_tokens
        self.model = model
        self.cache = {}  # prefix_hash -> (token_ids, ref_count, last_access)
        self.used_tokens = 0
        self.hits = 0
        self.misses = 0
        self.time_step = 0

    def _hash_prefix(self, token_ids: tuple) -> int:
        return hash(token_ids)

    def lookup(self, token_ids: list) -> Tuple[int, bool]:
        """Find longest matching prefix in cache. Returns (matched_len, is_hit)."""
        self.time_step += 1
        best_match = 0
        # Try progressively shorter prefixes
        for end in range(len(token_ids), 0, -1):
            prefix = tuple(token_ids[:end])
            h = self._hash_prefix(prefix)
            if h in self.cache:
                self.cache[h] = (prefix, self.cache[h][1] + 1, self.time_step)
                best_match = end
                self.hits += 1
                return best_match, True

        self.misses += 1
        return 0, False

    def insert(self, token_ids: list):
        """Insert a prefix into the cache."""
        prefix = tuple(token_ids)
        h = self._hash_prefix(prefix)
        if h not in self.cache:
            # Evict if needed
            while self.used_tokens + len(prefix) > self.capacity and self.cache:
                # LRU eviction
                oldest_key = min(self.cache.keys(),
                                key=lambda k: self.cache[k][2])
                self.used_tokens -= len(self.cache[oldest_key][0])
                del self.cache[oldest_key]
            self.cache[h] = (prefix, 1, self.time_step)
            self.used_tokens += len(prefix)

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0


# Simulate with shared system prompts
system_prompts = {
    'assistant': list(range(500)),    # 500-token system prompt
    'coder': list(range(500, 1200)),  # 700-token system prompt
    'analyst': list(range(1200, 1800)),  # 600-token system prompt
}

pool = KVCachePool(capacity_tokens=100000, model=llama_70b)

# Simulate 500 requests with varying system prompts
savings = []
for i in range(500):
    # Pick a random system prompt
    prompt_type = np.random.choice(list(system_prompts.keys()),
                                    p=[0.5, 0.3, 0.2])
    sys_prompt = system_prompts[prompt_type]
    user_query = list(range(10000, 10000 + np.random.randint(50, 300)))
    full_prompt = sys_prompt + user_query

    matched_len, hit = pool.lookup(sys_prompt)
    if hit:
        tokens_saved = matched_len
    else:
        tokens_saved = 0
        pool.insert(sys_prompt)

    savings.append({
        'request': i,
        'prompt_type': prompt_type,
        'total_len': len(full_prompt),
        'matched_len': matched_len,
        'saved_fraction': matched_len / len(full_prompt) if hit else 0,
    })

print(f"KV-Cache Pool Statistics:")
print(f"  Hit rate: {pool.hit_rate:.1%}")
print(f"  Cache entries: {len(pool.cache)}")
print(f"  Cache utilization: {pool.used_tokens / pool.capacity:.1%}")
avg_saved = np.mean([s['saved_fraction'] for s in savings])
print(f"  Avg tokens saved per request: {avg_saved:.1%}")

# Plot hit rate over time
cumulative_hits = np.cumsum([1 if s['matched_len'] > 0 else 0 for s in savings])
cumulative_rate = cumulative_hits / np.arange(1, len(savings) + 1)

plt.figure(figsize=(10, 4))
plt.plot(cumulative_rate, linewidth=2)
plt.xlabel('Request Number')
plt.ylabel('Cumulative Hit Rate')
plt.title('KV-Cache Prefix Hit Rate Over Time (Mooncake-style)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Assignment 1: Implement a Disaggregated Scheduler

Build a scheduler that assigns requests to prefill and decode nodes with the following features:
1. **SLO-aware scheduling**: Prioritize requests close to their TTFT deadline
2. **Load balancing**: Distribute work evenly across GPU pools
3. **Preemption**: Allow preempting decode for urgent prefill requests

In [ ]:
# ASSIGNMENT 1: Implement a Disaggregated Scheduler

@dataclass(order=True)
class PrioritizedRequest:
    priority: float
    request: Request = field(compare=False)
    slo_ttft: float = field(compare=False, default=0.5)  # 500ms TTFT SLO
    slo_tpot: float = field(compare=False, default=0.05)  # 50ms TPOT SLO


class DisaggregatedScheduler:
    """SLO-aware scheduler for disaggregated serving."""

    def __init__(self, num_prefill_gpus: int, num_decode_gpus: int,
                 model: ModelConfig, gpu: GPUConfig,
                 slo_ttft: float = 0.5, slo_tpot: float = 0.05):
        self.num_prefill = num_prefill_gpus
        self.num_decode = num_decode_gpus
        self.model = model
        self.gpu = gpu
        self.slo_ttft = slo_ttft
        self.slo_tpot = slo_tpot

        # TODO: Initialize GPU state tracking
        # Hint: Track available time for each prefill and decode GPU
        self.prefill_available = [0.0] * num_prefill_gpus
        self.decode_available = [0.0] * num_decode_gpus

        # TODO: Initialize request queues
        self.prefill_queue = []  # Priority queue
        self.pending_transfer = []  # Requests waiting for KV transfer

    def compute_priority(self, req: Request, current_time: float) -> float:
        """
        TODO: Compute scheduling priority.

        Higher priority = more urgent. Consider:
        - Time until SLO deadline (arrival_time + slo_ttft)
        - Request size (shorter prompts may be prioritized)
        - Queue wait time

        Returns: priority score (higher = schedule sooner)
        """
        # YOUR CODE HERE
        deadline = req.arrival_time + self.slo_ttft
        time_remaining = deadline - current_time
        # Hint: urgency increases as deadline approaches
        priority = -time_remaining  # Negative because heapq is min-heap
        return priority

    def assign_prefill_gpu(self, req: Request) -> int:
        """
        TODO: Select the best prefill GPU for a request.

        Consider:
        - Least loaded GPU
        - Data locality (if KV-cache prefix is cached)
        - Expected completion time

        Returns: GPU index
        """
        # YOUR CODE HERE
        return int(np.argmin(self.prefill_available))

    def assign_decode_gpu(self, req: Request) -> int:
        """
        TODO: Select the best decode GPU.

        Consider:
        - Available memory for KV-cache
        - Current batch size on each GPU
        - Load balancing

        Returns: GPU index
        """
        # YOUR CODE HERE
        return int(np.argmin(self.decode_available))

    def schedule(self, requests: List[Request]) -> List[Request]:
        """
        TODO: Schedule all requests through the disaggregated system.

        Steps:
        1. Sort requests by arrival time
        2. For each request, compute priority
        3. Assign to prefill GPU based on priority and load
        4. After prefill, handle KV-cache transfer
        5. Assign to decode GPU
        6. Track SLO violations

        Returns: List of completed requests with timing information
        """
        import copy
        results = [copy.deepcopy(r) for r in sorted(requests, key=lambda r: r.arrival_time)]
        slo_violations = 0

        # YOUR CODE HERE
        # Implement the full scheduling loop
        for req in results:
            # 1. Assign prefill GPU
            pf_gpu = self.assign_prefill_gpu(req)
            pf_start = max(req.arrival_time, self.prefill_available[pf_gpu])
            total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
            pf_time = 2 * total_params * req.prompt_len / (self.gpu.compute_tflops * 1e12)
            req.prefill_start = pf_start
            req.prefill_end = pf_start + pf_time
            self.prefill_available[pf_gpu] = req.prefill_end

            # 2. KV transfer time
            kv_size = self.model.kv_cache_per_token_bytes * req.prompt_len
            transfer_time = kv_size / (25 * 1e9)  # 25 GB/s InfiniBand

            # 3. Assign decode GPU
            dc_gpu = self.assign_decode_gpu(req)
            dc_start = max(req.prefill_end + transfer_time, self.decode_available[dc_gpu])
            bytes_to_load = total_params * self.model.param_bytes
            decode_per_token = bytes_to_load / (self.gpu.memory_bandwidth_gbps * 1e9)
            req.decode_start = dc_start
            req.decode_end = dc_start + decode_per_token * req.output_len
            self.decode_available[dc_gpu] = req.decode_end

            # 4. Check SLO
            if req.ttft > self.slo_ttft:
                slo_violations += 1

        print(f"SLO violations: {slo_violations}/{len(results)} "
              f"({slo_violations/len(results):.1%})")
        return results


# Test your scheduler
scheduler = DisaggregatedScheduler(2, 6, llama_70b, h100, slo_ttft=0.5)
scheduled_results = scheduler.schedule(workload)
metrics = compute_metrics(scheduled_results)
print(f"\nScheduler Results:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

---
## Assignment 2: Analyze Transfer Overhead vs Recomputation

Determine the conditions under which KV-cache transfer is preferable to recomputation on the decode node, considering:
1. Network bandwidth
2. Sequence length
3. Model size
4. GPU compute capability

In [ ]:
# ASSIGNMENT 2: Transfer Overhead vs Recomputation Analysis

def analyze_transfer_vs_recompute(models: List[ModelConfig],
                                   gpus: List[GPUConfig],
                                   bandwidths: List[float],
                                   seq_lengths: List[int]) -> dict:
    """
    TODO: For each combination of model, GPU, bandwidth, and sequence length,
    determine whether transfer or recomputation is better.

    Return a dictionary with:
    - 'crossover_points': For each (model, GPU, bandwidth) combo, the sequence
      length at which transfer becomes cheaper than recomputation.
    - 'recommendations': Human-readable recommendations.
    - 'data': Full comparison data for plotting.
    """
    results = {'crossover_points': {}, 'recommendations': [], 'data': []}

    # YOUR CODE HERE
    for model in models:
        sim = KVCacheTransferSimulator(model)
        for gpu in gpus:
            for bw in bandwidths:
                crossover = None
                for sl in range(64, 32769, 64):
                    t_transfer = sim.transfer_time(sl, bw)
                    t_recompute = sim.recompute_time(sl, gpu)
                    if t_transfer < t_recompute and crossover is None:
                        crossover = sl

                key = (model.name, gpu.name, bw)
                results['crossover_points'][key] = crossover

                if crossover:
                    results['recommendations'].append(
                        f"{model.name} on {gpu.name} @ {bw} GB/s: "
                        f"Transfer wins above seq_len={crossover}")
                else:
                    results['recommendations'].append(
                        f"{model.name} on {gpu.name} @ {bw} GB/s: "
                        f"Recomputation always wins")

    return results

# Run analysis
analysis = analyze_transfer_vs_recompute(
    models=[llama_13b, llama_70b],
    gpus=[a100, h100],
    bandwidths=[12.5, 25.0, 50.0, 100.0],
    seq_lengths=list(range(128, 8193, 128))
)

print("Transfer vs Recomputation Recommendations:")
print("=" * 60)
for rec in analysis['recommendations']:
    print(f"  {rec}")

In [ ]:
# Visualize crossover analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model in zip(axes, [llama_13b, llama_70b]):
    sim = KVCacheTransferSimulator(model)
    seq_range = np.arange(128, 8193, 64)

    for gpu, ls in [(a100, '--'), (h100, '-')]:
        recomp = [sim.recompute_time(s, gpu) * 1000 for s in seq_range]
        ax.plot(seq_range, recomp, f'k{ls}', linewidth=2,
                label=f'Recompute ({gpu.name})')

        for bw, color in [(12.5, 'blue'), (25.0, 'green'), (100.0, 'red')]:
            transfer = [sim.transfer_time(s, bw) * 1000 for s in seq_range]
            ax.plot(seq_range, transfer, color=color, linestyle=ls,
                    alpha=0.7, label=f'Transfer {bw}GB/s ({gpu.name})')

    ax.set_xlabel('Sequence Length')
    ax.set_ylabel('Time (ms)')
    ax.set_title(f'{model.name}: Transfer vs Recompute')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transfer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Assignment 3: Design a Disaggregated Architecture for Your Workload

Given a specific workload profile, design an optimal disaggregated serving architecture.

In [ ]:
# ASSIGNMENT 3: Architecture Design

@dataclass
class WorkloadProfile:
    name: str
    avg_prompt_len: int
    avg_output_len: int
    request_rate: float  # requests per second
    slo_ttft_ms: float
    slo_tpot_ms: float
    prompt_len_std: int = 200
    output_len_std: int = 100

# Define different workload profiles
workloads = [
    WorkloadProfile("Chatbot", 512, 256, 50, 500, 50),
    WorkloadProfile("Code Generation", 2048, 1024, 20, 1000, 30),
    WorkloadProfile("Summarization", 4096, 128, 30, 2000, 50),
    WorkloadProfile("RAG Pipeline", 8192, 256, 15, 3000, 50),
]


def design_architecture(profile: WorkloadProfile, model: ModelConfig,
                         gpu: GPUConfig, budget_gpus: int) -> dict:
    """
    TODO: Design an optimal disaggregated architecture.

    Return a dictionary with:
    - 'num_prefill_gpus': Number of GPUs for prefill
    - 'num_decode_gpus': Number of GPUs for decode
    - 'transfer_strategy': 'direct' or 'pipelined'
    - 'expected_throughput': Expected throughput
    - 'expected_slo_attainment': Fraction of requests meeting SLO
    - 'reasoning': Explanation of the design choices
    """
    # YOUR CODE HERE
    # Hint: Consider the ratio of prefill time to decode time
    # to determine the optimal split

    total_params = 12 * model.num_layers * model.hidden_dim ** 2

    # Estimate time per request for each phase
    prefill_time = 2 * total_params * profile.avg_prompt_len / (gpu.compute_tflops * 1e12)
    decode_time = (total_params * model.param_bytes / (gpu.memory_bandwidth_gbps * 1e9)) * profile.avg_output_len

    # Optimal ratio: proportional to phase duration
    total_time = prefill_time + decode_time
    prefill_fraction = prefill_time / total_time

    num_prefill = max(1, round(budget_gpus * prefill_fraction))
    num_decode = budget_gpus - num_prefill
    if num_decode < 1:
        num_decode = 1
        num_prefill = budget_gpus - 1

    # Simulate to estimate throughput
    test_workload = generate_workload(
        200, profile.request_rate,
        (max(64, profile.avg_prompt_len - profile.prompt_len_std),
         profile.avg_prompt_len + profile.prompt_len_std),
        (max(16, profile.avg_output_len - profile.output_len_std),
         profile.avg_output_len + profile.output_len_std)
    )

    server = DisaggregatedServer(num_prefill, num_decode, model, gpu)
    results = server.serve(test_workload)
    metrics = compute_metrics(results)

    slo_met = sum(1 for r in results if r.ttft <= profile.slo_ttft_ms / 1000) / len(results)

    return {
        'num_prefill_gpus': num_prefill,
        'num_decode_gpus': num_decode,
        'transfer_strategy': 'pipelined' if profile.avg_prompt_len > 2048 else 'direct',
        'expected_throughput': metrics['throughput'],
        'expected_slo_attainment': slo_met,
        'prefill_time_ms': prefill_time * 1000,
        'decode_time_ms': decode_time * 1000,
        'reasoning': (
            f"Prefill/decode time ratio: {prefill_fraction:.2f}/{1-prefill_fraction:.2f}. "
            f"Allocating {num_prefill} prefill + {num_decode} decode GPUs. "
            f"{'Pipelined' if profile.avg_prompt_len > 2048 else 'Direct'} transfer "
            f"recommended for avg prompt len={profile.avg_prompt_len}."
        )
    }


# Test your design for each workload
print(f"Architecture Recommendations (8 GPUs, LLaMA-70B, H100):")
print("=" * 80)
for wp in workloads:
    result = design_architecture(wp, llama_70b, h100, budget_gpus=8)
    print(f"\nWorkload: {wp.name}")
    print(f"  Configuration: {result['num_prefill_gpus']}P + {result['num_decode_gpus']}D")
    print(f"  Transfer: {result['transfer_strategy']}")
    print(f"  Expected throughput: {result['expected_throughput']:.2f} req/s")
    print(f"  SLO attainment: {result['expected_slo_attainment']:.1%}")
    print(f"  Reasoning: {result['reasoning']}")

## Summary

### Key Takeaways

1. **Phase disaggregation** exploits the fundamental difference between compute-bound prefill and memory-bound decode phases

2. **KV-cache transfer** is the critical path -- the choice between transfer and recomputation depends on network bandwidth, sequence length, and model size

3. **Optimal GPU ratio** depends on workload characteristics -- prefill-heavy workloads (long prompts, short outputs) need more prefill GPUs

4. **Prefix caching** (Mooncake-style) can dramatically reduce prefill costs when requests share common prefixes

5. **SLO-aware scheduling** is essential -- different phases have different latency targets (TTFT vs TPOT)

### Further Reading
- DistServe: https://arxiv.org/abs/2401.09670
- Splitwise: https://arxiv.org/abs/2311.18677
- Mooncake: https://arxiv.org/abs/2407.00079